# **Exercicio Duelo de Modelos 4**

Nesta tarefa, vocês irão criar o seu próprio duelo de modelos, com o objetivo de superar os resultados apresentados em aula. O desafio é alcançar um desempenho superior ao que obtivemos, e para isso, será necessário aplicar todas as melhorias que vocês aprenderam ao longo dos módulos, utilizando a base de dados do Titanic.

**1. Escolha do Modelo:**
Selecione um dos modelos que foram explorados nos duelos de modelos ao longo do curso. Pode ser SVM, Random Forest, XGBoost, ou qualquer outro que tenhamos abordado.

**2. Aperfeiçoamento:**
**Aplique as técnicas que aprendemos para melhorar o desempenho do seu modelo:**

**Hiperparâmetros:** Utilize GridSearchCV ou RandomSearchCV para encontrar os melhores parâmetros.

**Cross Validation:** Avalie a robustez do modelo utilizando validação cruzada para garantir que ele generaliza bem.

**Balanceamento de Classes:** Se o seu modelo lida com problemas de classes desbalanceadas, explore técnicas como SMOTE, undersampling ou oversampling.

**Padronização e Normalização:** Lembre-se de padronizar os dados, especialmente se for usar modelos que são sensíveis à escala das variáveis.

**3. Submissão no Kaggle:**
Treine o seu modelo com os dados de treino e gere as previsões para os dados de teste. Lembre-se de que o conjunto de teste não possui a variável alvo (y_test), pois a avaliação será feita com base nas submissões no Kaggle.
Submeta suas previsões na competição do Titanic no Kaggle.

**4. Entrega:**
Envie o código que você desenvolveu, detalhando cada etapa do seu processo de modelagem, explicando as escolhas feitas e como essas ajudaram a melhorar o modelo.

Junto com o código, envie um print do seu score obtido na plataforma do Kaggle. Esse score será a sua métrica final de avaliação, mostrando como o seu modelo se compara com os demais.

**5. Competição Saudável:**
A ideia é trazer um senso de competição saudável, então não vale replicar exatamente o que fizemos na aula! Inove, explore novas combinações de parâmetros e técnicas, e mostre do que é capaz. O importante é exercitar o pensamento crítico e a capacidade de experimentar.

**Dicas Finais:**

Seja criativo e tenha um olhar crítico sobre o que pode ser melhorado.
Teste diferentes abordagens e não se prenda a um único caminho.
Lembre-se de que, mais do que alcançar o melhor score, o objetivo é aprender e aplicar o conhecimento de forma prática e eficaz.
Boa sorte! Estamos ansiosos para ver como cada um de vocês vai se sair nesse desafio e quais insights irão surgir dessa competição!

Ao final dessa atividade vocês terão participado da primeira competição publica de ciência de dados de vocês = )




In [1]:
# import das bibliotecas
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import xgboost as xgb

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE


In [2]:
# bases de treino e teste
# traino tem o survived pois soa dados de treino e o test nao tem
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
# Verificando base treino
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Verificando base teste
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Verificar dados nulos no treino e no teste. O motivo é que o modelo aprende com o treino, entao se no treino e no test os dados nulos forem em colunas diferentes o modelo pode quebrar.

In [6]:
train_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
test_df.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [8]:
# Proporção de dados Survived
train_df['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Agora o ideal é entender quais variáveis parecem explicar a sobrevivência.
Algumas perguntas que devemos fazer:
Quem sobrevivei mais?
Mulheres ou Homens?
Classe 1, 2 ou 3 
Mulher da classe 1 sobreviveu igual a da classe 3 ?

In [9]:
# agrupamento por sex, considerando sobrevivencia
train_df.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [10]:
# agora a quantidade mais a media
train_df.groupby('Sex')['Survived'].agg(['count', 'mean'])

,count,mean
Sex,,
female,314,0.742038
male,577,0.188908


In [11]:
# agora pensar se a classe influencia na sobrevivencia

train_df.groupby('Pclass')['Survived'].agg(['count', 'mean'])

,count,mean
Pclass,,
1,216,0.629630
2,184,0.472826
3,491,0.242363


In [12]:
# agora pensando em classe x sex x survived
train_df.pivot_table(
    index="Sex",
    columns="Pclass",
    values="Survived",
    aggfunc="mean"
)

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [13]:
# mulher classe 1: 96.8% sobreviveu
# mulher classe 2: 92.1%
# mulher classe 3: 50.0%

# homem classe 1: 36.9%
# homem classe 2: 15.7%
# homem classe 3: 13.5%

In [14]:
# Precisamos descobrir s eo numero de psssoas na mesma familia
# interfere a sobrevivencia

In [15]:
train_df['family_size'] = train_df['SibSp'] + train_df['Parch'] + 1

In [16]:
train_df.groupby('family_size')['Survived'].agg(['count', 'mean'])

,count,mean
family_size,,
1,537,0.303538
2,161,0.552795
3,102,0.578431
4,29,0.724138
5,15,0.200000
6,22,0.136364
7,12,0.333333
8,6,0.000000
11,7,0.000000


In [17]:
# criar uma variavel de pessoas que estavam sozinhas

train_df['is_alone'] = train_df['family_size'] == 1
train_df.groupby('is_alone')['Survived'].agg(['count', 'mean'])

,count,mean
is_alone,,
False,354,0.505650
True,537,0.303538


In [18]:
# A pessoa tem cabine registrada ?
train_df['cabin_known'] = train_df['Cabin'].notna()
train_df.groupby('cabin_known')['Survived'].agg(['count', 'mean'])

,count,mean
cabin_known,,
False,687,0.299854
True,204,0.666667


In [19]:
# separar as cabines em sessoes, dentro do navio.
train_df['deck'] = train_df['Cabin'].str[0].fillna('Unknown')
train_df.groupby('deck')['Survived'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
deck,,
D,33,0.757576
E,32,0.750000
B,47,0.744681
F,13,0.615385
C,59,0.593220
G,4,0.500000
A,15,0.466667
Unknown,687,0.299854
T,1,0.000000


In [20]:
# titulos mostram status sociais, entao a hipotese é testar a sobrevivencia
train_df["title"] = train_df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
train_df['title'].value_counts()

title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [21]:
train_df.groupby('title')['Survived'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
title,,
the Countess,1,1.000000
Mlle,2,1.000000
Sir,1,1.000000
Ms,1,1.000000
Lady,1,1.000000
Mme,1,1.000000
Mrs,125,0.792000
Miss,182,0.697802
Master,40,0.575000


In [22]:
# limpeza dos dados de titulos
train_df['title_clean'] = train_df['title']

In [23]:
train_df['title_clean'] = train_df['title_clean'].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

In [24]:
# lista de prioritarios
main_title = ["Mr", "Mrs", "Miss", "Master"]

In [25]:
train_df['title_clean'] = train_df['title_clean'].where(
    train_df['title_clean'].isin(main_title),
    "Rare"
)

In [26]:
train_df['title_clean'].value_counts()

title_clean
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [27]:
train_df.groupby('title_clean')['Survived'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
title_clean,,
Mrs,126,0.793651
Miss,185,0.702703
Master,40,0.575000
Rare,23,0.347826
Mr,517,0.156673


In [28]:
# Análise da tarifa paga
train_df['fare_per_person'] = train_df['Fare'] / train_df['family_size']

In [29]:
train_df['fare_per_person'].describe()

count    891.000000
mean      19.916375
std       35.841257
min        0.000000
25%        7.250000
50%        8.300000
75%       23.666667
max      512.329200
Name: fare_per_person, dtype: float64

In [30]:
# A ideia é dividri o Fare por rupos e depois testar as sobrevivencia de acordo com o valor pago
train_df[ 'fare_per_person_bin'] = pd.qcut(
    train_df['fare_per_person'],
    q=4,
    labels=['low', 'mid_low', 'mid_high', 'high']
)

In [31]:
# analisando o agrupamentos por sobreviventes
train_df.groupby('fare_per_person_bin', observed=True)['Survived'].agg(['count', 'mean'])

,count,mean
fare_per_person_bin,,
low,226,0.265487
mid_low,220,0.254545
mid_high,223,0.408072
high,222,0.608108


In [32]:
# Crianças tinham prioridades ?
train_df['is_child'] = train_df['Age'] < 15
train_df.groupby('is_child')['Survived'].agg(['count', 'mean'])

,count,mean
is_child,,
False,813,0.365314
True,78,0.576923


In [33]:
# comparar faixa etaria apra sobrevivencia
def age_range(Age):
    if pd.isna(Age):
        return "Unknown"
    elif Age < 15:
        return "Child"
    elif Age < 30:
        return "Young"
    elif Age < 50:
        return "Adult"
    else:
        return "Senior"

In [34]:
train_df['age_range'] = train_df['Age'].apply(age_range)

In [35]:
train_df.groupby('age_range')['Survived'].agg(['count', 'mean'])

,count,mean
age_range,,
Adult,256,0.417969
Child,78,0.576923
Senior,74,0.364865
Unknown,177,0.293785
Young,306,0.362745


In [36]:
# Testar se pessoas com o mesmo ticket/viajaram juntas sobreviveram mais
train_df['ticket_group'] = train_df.groupby('Ticket')['PassengerId'].transform('count')

In [37]:
train_df.groupby('ticket_group')['Survived'].agg(['count', 'mean'])

,count,mean
ticket_group,,
1,547,0.297989
2,188,0.574468
3,63,0.698413
4,44,0.500000
5,10,0.000000
6,18,0.000000
7,21,0.238095


In [38]:
# versao categorica
def ticket_cat(ticket_group):
    if ticket_group == 1:
        return "Alone"
    elif ticket_group <= 4:
        return "SmallGroup"
    else:
        return "LargeGroup"

In [39]:
train_df['ticket_cat'] = train_df['ticket_group'].apply(ticket_cat)

In [40]:
train_df.groupby('ticket_cat')['Survived'].agg(['count', 'mean'])

,count,mean
ticket_cat,,
Alone,547,0.297989
LargeGroup,49,0.102041
SmallGroup,295,0.589831


In [41]:
# tentar extrari informaçoes do ticket prefixo nao numerico
def ticket_pre(Ticket):
    Ticket = str(Ticket)
    Ticket = str(Ticket).replace(".", "").replace("/", "").upper()
    partes = Ticket.split()

    if len(partes) > 1:
        return partes[0]
    else:
        return "NUMERIC"

In [42]:
train_df['ticket_pre'] = train_df['Ticket'].apply(ticket_pre)

In [43]:
train_df['ticket_pre'].value_counts()

ticket_pre
NUMERIC    665
PC          60
CA          41
A5          21
SOTONOQ     15
STONO       12
SCPARIS     11
WC          10
A4           7
STONO2       6
SOC          6
C            5
FCC          5
PP           3
WEP          3
SCAH         3
SOPP         3
SWPP         2
PPP          2
SOTONO2      2
SCA4         1
SP           1
SOP          1
FA           1
SCOW         1
SC           1
AS           1
FC           1
CASOTON      1
Name: count, dtype: int64

In [44]:
# analise de sobrevivencia
train_df.groupby('ticket_pre')['Survived'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
ticket_pre,,
SC,1,1.000000
SWPP,2,1.000000
FCC,5,0.800000
SCAH,3,0.666667
PP,3,0.666667
PC,60,0.650000
STONO2,6,0.500000
PPP,2,0.500000
SCPARIS,11,0.454545


In [45]:
# Agrupamento de ticket que aparecem menos

train_df['ticket_pre_clean'] = train_df['ticket_pre'] 

In [46]:
ticket_counts = train_df['ticket_pre_clean'].value_counts()
main_ticket = ticket_counts[ticket_counts >= 10].index

In [47]:
train_df['ticket_pre_clean'] = train_df['ticket_pre_clean'].where(
    train_df['ticket_pre_clean'].isin(main_ticket),
    "Rare"
)

In [48]:
train_df['ticket_pre_clean'].value_counts()

ticket_pre_clean
NUMERIC    665
PC          60
Rare        56
CA          41
A5          21
SOTONOQ     15
STONO       12
SCPARIS     11
WC          10
Name: count, dtype: int64

In [49]:
train_df.groupby('ticket_pre_clean')['Survived'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
ticket_pre_clean,,
PC,60,0.650000
SCPARIS,11,0.454545
STONO,12,0.416667
NUMERIC,665,0.383459
CA,41,0.341463
Rare,56,0.339286
SOTONOQ,15,0.133333
WC,10,0.100000
A5,21,0.095238


In [50]:
def make_features(df):
    df = df.copy()

    df["family_size"] = df["SibSp"] + df["Parch"] + 1
    df["is_alone"] = df["family_size"] == 1
    df["cabin_known"] = df["Cabin"].notna()
    df["deck"] = df["Cabin"].str[0].fillna("Unknown")

    df["title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)

    df["title_clean"] = df["title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    main_title = ["Mr", "Mrs", "Miss", "Master"]

    def clean_title(title):
        if title in main_title:
            return title
        else:
            return "Rare"

    df["title_clean"] = df["title_clean"].apply(clean_title)

    df["fare_per_person"] = df["Fare"] / df["family_size"]

    df["is_child"] = df["Age"] < 15
    df["age_range"] = df["Age"].apply(age_range)

    df["ticket_group"] = df.groupby("Ticket")["PassengerId"].transform("count")
    df["ticket_cat"] = df["ticket_group"].apply(ticket_cat)

    df["ticket_pre"] = df["Ticket"].apply(ticket_pre)

    return df

In [51]:
train_features = make_features(train_df)
test_features = make_features(test_df)

In [52]:
train_features

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,title,title_clean,fare_per_person,fare_per_person_bin,is_child,age_range,ticket_group,ticket_cat,ticket_pre,ticket_pre_clean
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,Mr,Mr,3.62500,low,False,Young,1,Alone,A5,A5
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,Mrs,Mrs,35.64165,high,False,Adult,1,Alone,PC,PC
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,Miss,Miss,7.92500,mid_low,False,Young,1,Alone,STONO2,Rare
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,Mrs,Mrs,26.55000,high,False,Adult,2,SmallGroup,NUMERIC,NUMERIC
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,Mr,Mr,8.05000,mid_low,False,Adult,1,Alone,NUMERIC,NUMERIC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,...,Rev,Rare,13.00000,mid_high,False,Young,1,Alone,NUMERIC,NUMERIC
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,...,Miss,Miss,30.00000,high,False,Young,1,Alone,NUMERIC,NUMERIC
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,...,Miss,Miss,5.86250,low,False,Unknown,2,SmallGroup,WC,WC
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,...,Mr,Mr,30.00000,high,False,Young,1,Alone,NUMERIC,NUMERIC


In [53]:
test_features

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,...,cabin_known,deck,title,title_clean,fare_per_person,is_child,age_range,ticket_group,ticket_cat,ticket_pre
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,...,False,Unknown,Mr,Mr,7.829200,False,Adult,1,Alone,NUMERIC
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,...,False,Unknown,Mrs,Mrs,3.500000,False,Adult,1,Alone,NUMERIC
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,...,False,Unknown,Mr,Mr,9.687500,False,Senior,1,Alone,NUMERIC
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,...,False,Unknown,Mr,Mr,8.662500,False,Young,1,Alone,NUMERIC
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,...,False,Unknown,Mrs,Mrs,4.095833,False,Young,1,Alone,NUMERIC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,...,False,Unknown,Mr,Mr,8.050000,False,Unknown,1,Alone,A5
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,...,True,C,Dona,Rare,108.900000,False,Adult,1,Alone,PC
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,...,False,Unknown,Mr,Mr,7.250000,False,Adult,1,Alone,SOTONOQ
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,...,False,Unknown,Mr,Mr,8.050000,False,Unknown,1,Alone,NUMERIC


In [54]:
# treinamento do modelo
features_cols = [
    "Pclass", "Sex", "Age", "Fare", "Embarked",
    "family_size", "is_alone", "cabin_known", "deck", "title_clean",
    "fare_per_person", "is_child", "age_range", "ticket_group",
    "ticket_cat", "ticket_pre"
]

x = train_features[features_cols]
y = train_features['Survived']

x_test = test_features[features_cols]

In [55]:
# Separaçao de colunas numericas e categoricas
numeric_features = [
    'Age',
    'Fare',
    'family_size',
    'fare_per_person',
    'ticket_group'
]

categorical_features = [
    'Pclass',
    'Sex',
    'Embarked',
    'is_alone',
    'cabin_known',
    'deck',
    'title_clean',
    'is_child',
    'age_range',
    'ticket_cat',
    'ticket_pre'
]

In [56]:
len(numeric_features) + len(categorical_features)

16

In [57]:
# imputar as nule com mediana
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [58]:
# imputar em nas categorias categoricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [59]:
# precisa aplicar os tranformers nas features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [60]:
# primeiro modelo de teste, regressao logistica
model_logistic = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=0))
])

In [61]:
#avaliacao do modelo em k folds
scores_logistic = cross_val_score(
    model_logistic,
    x, 
    y,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'accuracy'
)

In [62]:
scores_logistic

array([0.84916201, 0.8258427 , 0.79775281, 0.81460674, 0.82022472])

In [63]:
print("Accuracy média:", scores_logistic.mean())
print("Desvio padrão:", scores_logistic.std())

Accuracy média: 0.8215177954930638
Desvio padrão: 0.016715960796777434


LogisticRegression com preprocessing + KFold obteve accuracy média de 0.8215, acima do baseline por sexo de 0.7868.

In [64]:
# testes com SVM
svm_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC())
])

In [65]:
scores_SVM = cross_val_score(
    svm_model,
    x,
    y,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring= 'accuracy'
)

In [66]:
scores_SVM.mean()

np.float64(0.8327537505492435)

Baseline sexo:       0.7868
LogisticRegression:  0.8215
SVM:                 0.8328

In [67]:
# test com random forest
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))
])

In [68]:
scores_rf = cross_val_score(
    rf_model,
    x,
    y,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'accuracy'
)

In [69]:
scores_rf.mean()

np.float64(0.8305128366078713)

1. SVM:          0.8328
2. RandomForest: 0.8305
3. Logistic:     0.8215
4. Baseline:     0.7868

In [70]:
# testando XGboost
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))
])

In [71]:
scores_xgb = cross_val_score(
    xgb_model,
    x,
    y,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'accuracy'
)

In [72]:
scores_xgb

array([0.8603352 , 0.80337079, 0.7752809 , 0.83707865, 0.8258427 ])

A nossa baseline é 78 por sexo, entao ficaria assim
1. Baseline sexo:       0.7868
2. LogisticRegression:  0.8215
3. XGBoost:             0.8204
4. RandomForest:        0.8305
5. SVM:                 0.8328

In [73]:
# grid search no SVM que foi o melhor desempenho
param_grid_svm = {
    "model__C": [0.1, 1, 3, 10],
    "model__kernel": ["rbf"],
    "model__gamma": ["scale", 0.01, 0.03, 0.1]
}

grid_svm = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid_svm,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'accuracy'
)

grid_svm.fit(x,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 0.01, ...], 'model__kernel': ['rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,

In [74]:
grid_svm.best_score_

np.float64(0.8350009415604795)

In [75]:
grid_svm.best_params_

{'model__C': 3, 'model__gamma': 0.03, 'model__kernel': 'rbf'}

GridSearch no SVM melhorou a accuracy média de 0.8328 para 0.8350. Melhor combinação: C=3, gamma=0.03, kernel=rbf.

In [76]:
# teste de modelo com grid search no random forest
param_grid_rf = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [4, 6, 8, 10, None],
    'model__min_samples_leaf': [1, 2, 4, 7]
}

grid_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'accuracy'
)

grid_rf.fit(x,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [4, 6, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [100, 300, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for cal

In [77]:
grid_rf.best_score_

np.float64(0.8383717280773334)

In [78]:
grid_rf.best_params_

{'model__max_depth': 10,
 'model__min_samples_leaf': 4,
 'model__n_estimators': 300}

1. Baseline sexo:       0.7868
2. LogisticRegression:  0.8215
3. XGBoost base:        ~0.8204
4. SVM base:            0.8328
5. SVM GridSearch:      0.8350
6. RandomForest base:   0.8305
7. RandomForest Grid:   0.8384

In [79]:
# Teste de XGBoost com hiperparametros
param_xgb = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [2, 3, 4, 5],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__subsample': [0.7, 0.85, 1.0],
    'model__colsample_bytree': [0.7, 0.85, 1.0],
    'model__min_child_weight': [1, 2, 3, 5]
}

random_xgb = RandomizedSearchCV(
    estimator = xgb_model,
    param_distributions = param_xgb,
    n_iter = 30,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring= 'accuracy',
    random_state=42
)

random_xgb.fit(x,y)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': [0.7, 0.85, ...], 'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [2, 3, ...], 'model__min_child_weight': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits usin

In [80]:
random_xgb.best_score_

np.float64(0.8451007469713139)

In [81]:
random_xgb.best_params_

{'model__subsample': 0.7,
 'model__n_estimators': 200,
 'model__min_child_weight': 3,
 'model__max_depth': 4,
 'model__learning_rate': 0.1,
 'model__colsample_bytree': 0.85}

ranking dos modelos
1. Baseline sexo:       0.7868
2. LogisticRegression:  0.8215
3. SVM base:            0.8328
4. SVM GridSearch:      0.8350
5. RandomForest Grid:   0.8384
6. XGBoost tunado:      0.8451

In [82]:
# treinar o melhor modelo e fazer os testes
best_model = random_xgb.best_estimator_
best_model.fit(x,y)
predictions = best_model.predict(x_test)

In [83]:
# montar submissao
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions
})

In [84]:
#salvar o CSV
submission.to_csv('submission.csv', index=False)

In [85]:
# conferindo
submission.head()
submission.shape
submission['Survived'].value_counts()

Survived
0    262
1    156
Name: count, dtype: int64

## Submissoes alternativas que melhoraram o Kaggle

Depois das primeiras submissões, os modelos mais complexos tiveram CV local alto, mas score Kaggle baixo. Para investigar, criamos duas alternativas mais conservadoras:

1. `submission_rf_simple.csv`: RandomForest com menos features, evitando `ticket_pre`, `ticket_group` e `deck`.
2. `submission_gender_family_rules.csv`: baseline por sexo com ajustes fortes por família/ticket observados no treino.


### RandomForest simples

Ideia: reduzir overfit usando apenas features estáveis. O modelo usa `Pclass`, `Sex`, `Age`, `Fare`, `Embarked`, `family_size` e `title_clean`.

In [86]:
# RandomForest simples para submissao conservadora

def make_simple_features(df):
    df = df.copy()
    df["family_size"] = df["SibSp"] + df["Parch"] + 1
    df["title_clean"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
    df["title_clean"] = df["title_clean"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })
    df["title_clean"] = df["title_clean"].where(
        df["title_clean"].isin(["Mr", "Mrs", "Miss", "Master"]),
        "Rare"
    )
    return df

train_simple = make_simple_features(train_df)
test_simple = make_simple_features(test_df)

simple_features_cols = [
    "Pclass", "Sex", "Age", "Fare", "Embarked", "family_size", "title_clean"
]

simple_numeric_features = ["Age", "Fare", "family_size"]
simple_categorical_features = ["Pclass", "Sex", "Embarked", "title_clean"]

simple_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

simple_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

simple_preprocessor = ColumnTransformer(transformers=[
    ("num", simple_numeric_transformer, simple_numeric_features),
    ("cat", simple_categorical_transformer, simple_categorical_features)
])

rf_simple_model = Pipeline(steps=[
    ("preprocessor", simple_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        random_state=42
    ))
])

rf_simple_model.fit(train_simple[simple_features_cols], train_simple["Survived"])
rf_simple_predictions = rf_simple_model.predict(test_simple[simple_features_cols])

submission_rf_simple = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": rf_simple_predictions.astype(int)
})

submission_rf_simple.to_csv("submission_rf_simple.csv", index=False)
submission_rf_simple.shape

(418, 2)

### Regra conservadora por sexo, família e ticket

Ideia: começar com o baseline por sexo e mudar apenas casos com evidência forte no treino. Isso reduziu overfit e foi a melhor submissão até aqui.

In [87]:
# Modelo por regras: baseline por sexo + sinais fortes de familia/ticket

def get_surname(name):
    return name.split(",")[0].strip()


def add_family_key(df):
    df = df.copy()
    df["family_size"] = df["SibSp"] + df["Parch"] + 1
    df["surname"] = df["Name"].apply(get_surname)
    df["family_key"] = df["surname"] + "_" + df["family_size"].astype(str)
    return df

train_rules = add_family_key(train_df)
test_rules = add_family_key(test_df)

family_stats = train_rules.groupby("family_key")["Survived"].agg(["count", "mean"])
ticket_stats = train_rules.groupby("Ticket")["Survived"].agg(["count", "mean"])

rule_predictions = (test_rules["Sex"] == "female").astype(int)
rule_changes = []

for idx, row in test_rules.iterrows():
    family_key = row["family_key"]
    ticket = row["Ticket"]
    sex = row["Sex"]
    age = row["Age"]
    title = row["Name"].split(",")[1].split(".")[0].strip()

    family_info = family_stats.loc[family_key] if family_key in family_stats.index else None
    ticket_info = ticket_stats.loc[ticket] if ticket in ticket_stats.index else None

    if sex == "female":
        if family_info is not None and family_info["count"] >= 2 and family_info["mean"] == 0:
            rule_predictions.iloc[idx] = 0
            rule_changes.append((row["PassengerId"], "female_dead_family"))
        elif ticket_info is not None and ticket_info["count"] >= 2 and ticket_info["mean"] == 0:
            rule_predictions.iloc[idx] = 0
            rule_changes.append((row["PassengerId"], "female_dead_ticket"))
    else:
        is_child_signal = title == "Master" or (pd.notna(age) and age < 15)
        if is_child_signal and family_info is not None and family_info["count"] >= 2 and family_info["mean"] == 1:
            rule_predictions.iloc[idx] = 1
            rule_changes.append((row["PassengerId"], "male_alive_family"))
        elif is_child_signal and ticket_info is not None and ticket_info["count"] >= 2 and ticket_info["mean"] == 1:
            rule_predictions.iloc[idx] = 1
            rule_changes.append((row["PassengerId"], "male_alive_ticket"))

submission_gender_family_rules = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": rule_predictions.astype(int)
})

submission_gender_family_rules.to_csv("submission_gender_family_rules.csv", index=False)
print("Alteracoes sobre baseline por sexo:", len(rule_changes))
submission_gender_family_rules.shape

Alteracoes sobre baseline por sexo: 13


(418, 2)

In [88]:
# Validacao final dos arquivos de submissao
for file_name in ["submission_rf_simple.csv", "submission_gender_family_rules.csv"]:
    submission_check = pd.read_csv(file_name)
    print(file_name)
    print("shape:", submission_check.shape)
    print("colunas:", list(submission_check.columns))
    print("valores Survived:", sorted(submission_check["Survived"].unique()))
    print("nulos:", submission_check.isna().sum().sum())
    print()

submission_rf_simple.csv
shape: (418, 2)
colunas: ['PassengerId', 'Survived']
valores Survived: [np.int64(0), np.int64(1)]
nulos: 0

submission_gender_family_rules.csv
shape: (418, 2)
colunas: ['PassengerId', 'Survived']
valores Survived: [np.int64(0), np.int64(1)]
nulos: 0



Resultado no Kaggle:

- `submission_gender_family_rules.csv`: `0.79186`
- `submission_rf_simple.csv`: `0.76555`

A melhor submissão até aqui foi a regra conservadora por sexo, família e ticket.

Conclusão:

Antes de avaliar modelos complexos, é importante criar um baseline simples. No caso do Titanic, a variável Sex é muito forte, porque mulheres tiveram uma taxa de sobrevivência muito maior que homens.

A regra simples "prever que todas as mulheres sobreviveram e todos os homens não sobreviveram" já gera uma accuracy alta. Esse resultado funciona como nota de corte mínima.

Se um modelo de machine learning tiver desempenho abaixo desse baseline, ele não está agregando valor. Isso significa que, apesar de ser mais complexo, ele está tomando decisões piores do que uma regra simples baseada apenas em sexo.

Portanto, modelos com accuracy inferior ao baseline por sexo devem ser considerados ruins para este problema, pois aumentam a complexidade sem melhorar a capacidade preditiva.

## Candidatos conservadores para tentar superar 0.80

O melhor score público até aqui foi `0.79186` com baseline por sexo + ajustes fortes por família/ticket. Como os modelos complexos ficaram piores no Kaggle, a próxima busca é testar pequenas variações conservadoras.

A hipótese é simples: discordar do baseline por sexo apenas quando há evidência forte no treino. Cada arquivo muda poucos passageiros, permitindo medir no Kaggle qual regra realmente ajuda.


In [89]:
# Funcoes para gerar variantes conservadoras de submissao

def surname_from_name(name):
    return name.split(",")[0].strip()


def title_from_name(name):
    try:
        return name.split(",")[1].split(".")[0].strip()
    except Exception:
        return ""


def enrich_family_ticket(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["Surname"] = df["Name"].apply(surname_from_name)
    df["Title"] = df["Name"].apply(title_from_name)
    df["FamilyKey"] = df["Surname"] + "_" + df["FamilySize"].astype(str)
    return df


train_rules = enrich_family_ticket(train_df)
test_rules = enrich_family_ticket(test_df)

family_stats = train_rules.groupby("FamilyKey")["Survived"].agg(["count", "mean"])
ticket_stats = train_rules.groupby("Ticket")["Survived"].agg(["count", "mean"])
base_gender_predictions = (test_rules["Sex"] == "female").astype(int)


def create_rule_submission(
    file_name,
    female_family=True,
    female_ticket=True,
    male_family=True,
    male_ticket=True,
    min_count=2,
    female_only_p3=False,
    male_master_only=True,
):
    predictions = base_gender_predictions.copy()
    changes = []

    for idx, row in test_rules.iterrows():
        family_info = family_stats.loc[row["FamilyKey"]] if row["FamilyKey"] in family_stats.index else None
        ticket_info = ticket_stats.loc[row["Ticket"]] if row["Ticket"] in ticket_stats.index else None

        if row["Sex"] == "female":
            if female_only_p3 and row["Pclass"] != 3:
                continue
            if female_family and family_info is not None and family_info["count"] >= min_count and family_info["mean"] == 0:
                predictions.iloc[idx] = 0
                changes.append((row["PassengerId"], "female_dead_family"))
            elif female_ticket and ticket_info is not None and ticket_info["count"] >= min_count and ticket_info["mean"] == 0:
                predictions.iloc[idx] = 0
                changes.append((row["PassengerId"], "female_dead_ticket"))
        else:
            is_child_signal = row["Title"] == "Master" if male_master_only else row["Title"] == "Master" or (pd.notna(row["Age"]) and row["Age"] < 15)
            if is_child_signal and male_family and family_info is not None and family_info["count"] >= min_count and family_info["mean"] == 1:
                predictions.iloc[idx] = 1
                changes.append((row["PassengerId"], "male_alive_family"))
            elif is_child_signal and male_ticket and ticket_info is not None and ticket_info["count"] >= min_count and ticket_info["mean"] == 1:
                predictions.iloc[idx] = 1
                changes.append((row["PassengerId"], "male_alive_ticket"))

    submission_candidate = pd.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": predictions.astype(int)
    })
    submission_candidate.to_csv(file_name, index=False)
    return file_name, submission_candidate, changes


In [90]:
# Gerar candidatos para submissao Kaggle
rule_candidates = []

rule_candidates.append(create_rule_submission(
    "sub_rules_family_only.csv",
    female_family=True, female_ticket=False, male_family=True, male_ticket=False
))

rule_candidates.append(create_rule_submission(
    "sub_rules_female_family_only.csv",
    female_family=True, female_ticket=False, male_family=False, male_ticket=False
))

rule_candidates.append(create_rule_submission(
    "sub_rules_female_all_dead.csv",
    female_family=True, female_ticket=True, male_family=False, male_ticket=False
))

rule_candidates.append(create_rule_submission(
    "sub_rules_male_master_alive_only.csv",
    female_family=False, female_ticket=False, male_family=True, male_ticket=True
))

rule_candidates.append(create_rule_submission(
    "sub_rules_strict_count3.csv",
    female_family=True, female_ticket=True, male_family=True, male_ticket=True, min_count=3
))

for file_name, submission_candidate, changes in rule_candidates:
    print(file_name, submission_candidate.shape, submission_candidate["Survived"].value_counts().sort_index().to_dict(), "changes:", len(changes))


sub_rules_family_only.csv (418, 2) {0: 272, 1: 146} changes: 10
sub_rules_female_family_only.csv (418, 2) {0: 274, 1: 144} changes: 8
sub_rules_female_all_dead.csv (418, 2) {0: 275, 1: 143} changes: 9
sub_rules_male_master_alive_only.csv (418, 2) {0: 263, 1: 155} changes: 3
sub_rules_strict_count3.csv (418, 2) {0: 271, 1: 147} changes: 5


Ordem sugerida para testar no Kaggle:

1. `sub_rules_family_only.csv`
2. `sub_rules_female_family_only.csv`
3. `sub_rules_female_all_dead.csv`
4. `sub_rules_strict_count3.csv`
5. `sub_rules_male_master_alive_only.csv`

A melhor atual continua sendo `submission_gender_family_rules.csv` com `0.79186` até uma dessas variações superar.

### Auditoria das mudanças dos candidatos

Esta tabela mostra quais passageiros cada candidato mudou em relação ao baseline por sexo. Ao receber o score Kaggle de cada arquivo, usamos esta auditoria para identificar quais alterações provavelmente ajudaram ou atrapalharam.

In [91]:
# Auditoria: quais passageiros cada candidato altera vs baseline por sexo
candidate_files = [
    "submission_gender_family_rules.csv",
    "sub_wc_family_surnamefare.csv",
    "sub_wc_surnamefare_dead_only.csv",
    "sub_wc_all_keys_dead_only.csv",
    "sub_wc_strict3_all_keys.csv",
    "sub_adult_male_survivor_groups.csv",
]

gender_baseline = gender_submission.copy() if "gender_submission" in globals() else pd.read_csv("gender_submission.csv")
gender_baseline = gender_baseline.rename(columns={"Survived": "gender_pred"})
base_audit = test_df.merge(gender_baseline, on="PassengerId")

change_rows = []
for file_name in candidate_files:
    candidate_submission = pd.read_csv(file_name).rename(columns={"Survived": "pred"})
    audit_df = base_audit.merge(candidate_submission, on="PassengerId")
    changed = audit_df[audit_df["pred"] != audit_df["gender_pred"]]

    for _, row in changed.iterrows():
        change_rows.append({
            "file": file_name,
            "PassengerId": row["PassengerId"],
            "change": "female->dead" if row["Sex"] == "female" else "male->alive",
            "Sex": row["Sex"],
            "Pclass": row["Pclass"],
            "Name": row["Name"],
            "Age": row["Age"],
            "SibSp": row["SibSp"],
            "Parch": row["Parch"],
            "Ticket": row["Ticket"],
            "Fare": row["Fare"],
            "Cabin": row["Cabin"],
        })

candidate_change_ledger = pd.DataFrame(change_rows)
candidate_change_ledger.to_csv("candidate_change_ledger.csv", index=False)

candidate_change_ledger.groupby("file").size().sort_values()

file
sub_wc_strict3_all_keys.csv            5
sub_wc_surnamefare_dead_only.csv       8
sub_wc_all_keys_dead_only.csv          9
sub_wc_family_surnamefare.csv         12
submission_gender_family_rules.csv    13
sub_adult_male_survivor_groups.csv    17
dtype: int64

In [92]:
candidate_change_ledger.groupby([
    "PassengerId", "Name", "Sex", "Pclass", "change"
]).size().sort_values(ascending=False).head(30)

PassengerId  Name                                               Sex     Pclass  change      
1024         Lefebre, Mrs. Frank (Frances)                      female  3       female->dead    6
1032         Goodwin, Miss. Jessie Allis                        female  3       female->dead    6
1257         Sage, Mrs. John (Annie Bullen)                     female  3       female->dead    6
1080         Sage, Miss. Ada                                    female  3       female->dead    6
925          Johnston, Mrs. Andrew G (Elizabeth Lily" Watson)"  female  3       female->dead    5
929          Cacic, Miss. Manda                                 female  3       female->dead    5
1176         Rosblom, Miss. Salli Helena                        female  3       female->dead    5
1172         Oreskovic, Miss. Jelka                             female  3       female->dead    5
1259         Riihivouri, Miss. Susanna Juhantytar Sanni""       female  3       female->dead    4
1122         Sweet, Mr. G